In [ ]:
! pip install -q transformers datasets evaluate accelerate huggingface_hub rouge_score

Summarization can be:

- Extractive: extract the most relevant information from a document.
- Abstractive: generate new text that captures the most relevant information. (this notebook)

## Load dependencies

In [ ]:
import numpy as np
import evaluate
from huggingface_hub import notebook_login
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    pipeline
)

## Config

In [ ]:
user_name = "amin-oj"
dataset_dict = {"path": "billsum", "split": "ca_test"}
model_name = "google-t5/t5-small"
push_to_hub=True
train_bs = 16
eval_bs = 16
num_train_epochs = 3
task = "summarization"
ckpt_name = f"{model_name.split("/")[-1]}-finetuned-{task}-{dataset_dict["path"]}"

## HF Login

In [ ]:
notebook_login()

## Load dataset

In [ ]:
billsum = load_dataset(**dataset_dict)
billsum = billsum.train_test_split(test_size=0.2)
for k,v in billsum["train"][0].items():
    print("================")
    print(f"{k}: {v[:100]}")

There are two fields that you'll want to use:

- `text`: the text of the bill which'll be the input to the model.
- `summary`: a condensed version of `text` which'll be the model target.

## Preprocess

In [ ]:
def preprocess_function(examples, prefix, tokenizer):
    # Prompt the model with a prefix
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(
        inputs,
        max_length=1024,
        truncation=True
    )

    # Use the keyword `text_target` argument when tokenizing labels.
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_billsum = billsum.map(
    preprocess_function,
    batched=True,
    num_proc=2,
    fn_kwargs={"prefix": "summarize: ", "tokenizer": tokenizer},
    remove_columns=billsum["train"].column_names,
)

## Train

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_name)
# It's more efficient to *dynamically pad* the sentences to the longest length
# in a batch during collation, instead of padding the whole dataset to
# the maximum length.

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def compute_metrics(eval_pred):
    rouge = evaluate.load("rouge")
    predictions, labels = eval_pred
    # prediction are token_ids because we set predict_with_generate=True in the trainer object
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=ckpt_name,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    num_train_epochs=num_train_epochs,
    push_to_hub=push_to_hub,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    # tells the trainer to use the model's generate() method to create text predictions during evaluation,
    # rather than returning raw logits. This is essential for evaluating text generation tasks
    # using metrics like BLEU or ROUGE
    fp16=True,
    report_to = 'none' # to disable w&b
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_billsum["train"],
    eval_dataset=tokenized_billsum["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
if push_to_hub: trainer.push_to_hub()

## Inference

In [ ]:
# For T5, you need to prefix your input depending on the task you're working on.
text = """
summarize: The Inflation Reduction Act lowers prescription drug costs,
health care costs, and energy costs. It's the most aggressive action
on tackling the climate crisis in American history, which will lift up
American workers and create good-paying, union jobs across the country.
It'll lower the deficit and ask the ultra-wealthy and corporations
to pay their fair share. And no one making under $400,000 per year
will pay a penny more in taxes."""

### The simplest way

In [ ]:
summarizer = pipeline(task, model=f"{user_name}/{ckpt_name}")
summarizer(text)

### More complex | More flexible

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f"{user_name}/{ckpt_name}")
inputs = tokenizer(text, return_tensors="pt").input_ids
model = AutoModelForSeq2SeqLM.from_pretrained(f"{user_name}/{ckpt_name}")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)
tokenizer.decode(outputs[0], skip_special_tokens=True)